In [ ]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date

# =============================================================================
# PARTIE 1 : FONCTIONS MATHÉMATIQUES DE BASE
# =============================================================================
def Phi(x):
    """Fonction de répartition de la loi normale."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def black76_call(F, K, T, sigma):
    """Formule de Black-76 pour un Call."""
    d1 = (math.log(F / K) + 0.5 * sigma**2 * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return F * Phi(d1) - K * Phi(d2)

def black76_put(F, K, T, sigma):
    d1 = (math.log(F / K) + 0.5 * sigma**2 * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return K * Phi(-d2) - F * Phi(-d1)

def vol_implicite(prix, F, K, T):
    """Recherche de la volatilité par dichotomie."""
    bas, haut = 0.0001, 5.0
    for _ in range(100):  # Sécurité anti-boucle infinie (remplace le while)
        milieu = (bas + haut) / 2
        if black76_call(F, K, T, milieu) < prix:
            bas = milieu
        else:
            haut = milieu
        if haut - bas < 1e-6:
            break
    return (bas + haut) / 2

# =============================================================================
# PARTIE 2 : LECTURE DES DONNÉES ET CALIBRATION DES SMILES (6 janvier 2020)
# =============================================================================
# 1. Lecture du fichier
df = pd.read_parquet(r'C:\Users\Adèle\Documents\POLIMI\COMPUTATIONAL FINANCE\full_df_train.parquet')
d = df[df['date'] == '2020-01-06']
c = d[d['underlying_ticker'] == 'COJ21 Comdty']   # Avril
c_K = d[d['underlying_ticker'] == 'COK21 Comdty'] # Mai

# 1bis. Calcul de rho
fut = df[['date', 'underlying_ticker', 'underlying_settlement']].drop_duplicates()
J = fut[fut['underlying_ticker'] == 'COJ21 Comdty'].set_index('date')['underlying_settlement']
K = fut[fut['underlying_ticker'] == 'COK21 Comdty'].set_index('date')['underlying_settlement']
rJ = np.log(J / J.shift(1)).dropna()
rK = np.log(K / K.shift(1)).dropna()
rend = pd.concat([rJ, rK], axis=1, keys=['J', 'K']).dropna()
rho = rend['J'].iloc[:60].corr(rend['K'].iloc[:60])
print("rho :", rho)

# 2. Extraction des prix Forward actuels
f1 = c['underlying_settlement'].iloc[0]
f2 = c_K['underlying_settlement'].iloc[0]
rho = 0.9998 # Corrélation calculée (au lieu de 0.85)

# 3. Traitement COJ21 (Avril)
strikes, vols = [], []
for _, ligne in c.iterrows():
    if ligne['option_type'] == 'Call' and ligne['option_bid'] > 0:
        K = ligne['strike']
        prix = (ligne['option_bid'] + ligne['option_ask']) / 2
        F = ligne['underlying_settlement']
        T = ligne['time_to_expiry_days'] / 365
        strikes.append(K)
        vols.append(vol_implicite(prix, F, K, T))

coef = np.polyfit(strikes, vols, 3)

# 4. Traitement COK21 (Mai)
strikes_K, vols_K = [], []
for _, ligne in c_K.iterrows():
    if ligne['option_type'] == 'Call' and ligne['option_bid'] > 0:
        K = ligne['strike']
        prix = (ligne['option_bid'] + ligne['option_ask']) / 2
        F = ligne['underlying_settlement']
        T = ligne['time_to_expiry_days'] / 365
        strikes_K.append(K)
        vols_K.append(vol_implicite(prix, F, K, T))

coef_K = np.polyfit(strikes_K, vols_K, 3)

# =============================================================================
# PARTIE 3 : PRICING DU CALL ASIATIQUE (MÉTHODE DES MOMENTS)
# =============================================================================
# Étape 1 : Gestion du temps (thetas) pour les jours de fixing de février 2021
date_depart = date(2020, 1, 6)
thetas = []
for jour in range(1, 29): 
    dt = date(2021, 2, jour)
    if dt.weekday() < 5:  # Lundi à Vendredi uniquement
        thetas.append((dt - date_depart).days / 365.0)

n_total = len(thetas)
n1 = 18
n2 = n_total - n1

theta_1 = thetas[:n1]  
theta_2 = thetas[n1:]  

# Étape 2 : Calcul du Forward global du panier (F_avg)
w1 = n1 / n_total
w2 = n2 / n_total
F_avg = w1 * f1 + w2 * f2

# Étape 3 : Projection des strikes (Sticky-Moneyness)
K_call = 78.0
K1_call = f1 * (K_call / F_avg)
K2_call = f2 * (K_call / F_avg)

# Étape 4 : Lecture des volatilités sur les polynômes (coef)
a1, b1, c1, d1 = coef
sigma1 = a1 * (K1_call**3) + b1 * (K1_call**2) + c1 * K1_call + d1

a2, b2, c2, d2 = coef_K
sigma2 = a2 * (K2_call**3) + b2 * (K2_call**2) + c2 * K2_call + d2

# Étape 5 : Méthode des moments (Calcul des variances et covariance)
def variance_totale(sigma1, sigma2, theta_1, theta_2, f1, f2, w1, w2, n1, n2):
    var_A1 = sum((2 * (n1 - k) + 1) * (math.exp(sigma1**2 * theta_1[k-1]) - 1) for k in range(1, n1 + 1))
    var_A1 *= (f1**2 / n1**2)

    var_A2 = sum((2 * (n2 - j) + 1) * (math.exp(sigma2**2 * theta_2[j-1]) - 1) for j in range(1, n2 + 1))
    var_A2 *= (f2**2 / n2**2)

    cov = sum(math.exp(rho * sigma1 * sigma2 * theta_1[k-1]) - 1 for k in range(1, n1 + 1))
    cov *= (f1 * f2 / n1)

    var_totale = (w1**2 * var_A1) + (w2**2 * var_A2) + (2 * w1 * w2 * cov)
    return var_totale

var_totale = variance_totale(sigma1, sigma2, theta_1, theta_2, f1, f2, w1, w2, n1, n2)
v = math.sqrt(math.log(1 + (var_totale / (F_avg**2))))

T_final = 1.0 # Le paramètre v intègre déjà le temps, on passe T=1.0 à Black-76
prix_call = black76_call(F_avg, K_call, T_final, v)

# =============================================================================
# PARTIE 4 : Kput
# =============================================================================
# On cherche K_put entre 40$ (très bas) et 78$ (notre Call)
bas = 40.0
haut = 78.0
K_put = (bas + haut) / 2
prix_put = 0  # Initialisation

# Sécurité anti-boucle infinie 
for _ in range(100):
    
    # 1. On teste le milieu exact de notre fourchette
    K_put = (bas + haut) / 2
    
    # 2. On recalcule tout avec ce K_put testé
    K_put_1 = f1 * (K_put / F_avg)
    K_put_2 = f2 * (K_put / F_avg)
    
    sigma_put_1 = a1 * (K_put_1**3) + b1 * (K_put_1**2) + c1 * K_put_1 + d1
    sigma_put_2 = a2 * (K_put_2**3) + b2 * (K_put_2**2) + c2 * K_put_2 + d2
    
    var_totale_put = variance_totale(sigma_put_1, sigma_put_2, theta_1, theta_2, f1, f2, w1, w2, n1, n2)
    v_put = math.sqrt(math.log(1 + (var_totale_put / (F_avg**2))))
    
    # ATTENTION : On utilise bien black76_put ici et T_final = 1.0 pour pas prendre le temps 2 fois en comtpe
    prix_put = black76_put(F_avg, K_put, T_final, v_put)
    
    # 3. L'intelligence de la dichotomie : on ajuste la fourchette
    if prix_put > prix_call:
        # Le Put est trop cher, il faut baisser le strike
        haut = K_put
    else:
        # Le Put n'est pas assez cher, il faut monter le strike
        bas = K_put
        
    if abs(prix_put - prix_call) < 0.0001:
        break

print(f"Strike exact du Put pour un Zero-Cost Collar : {K_put:.2f} $")
print(f"Vérification - Prix du Call : {prix_call:.4f} $ | Prix du Put : {prix_put:.4f} $")

# =============================================================================
# RÉSULTATS
# =============================================================================
print("--- RÉSULTATS DU PRICING (6 JANVIER 2020) ---")
print(f"Prix Forward du panier (F_avg) : {F_avg:.2f} $")
print(f"Volatilité globale du panier (v) : {v:.4f}")
print(f"Prix théorique du Call Asiatique à 78$ : {prix_call:.4f} $")
print(f"Prime totale du Call (pour 1 000 000 barils) : {prix_call * 1_000_000:,.2f} $")

# Affichage des graphes optionnel à la fin
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
x = np.linspace(min(strikes), max(strikes), 100)
plt.plot(strikes, vols, 'o', label='Vols implicites')
plt.plot(x, np.polyval(coef, x), 'r-', label='Courbe ajustée')
plt.title('Smile COJ21 (Avril)')
plt.legend()

plt.subplot(1, 2, 2)
x_K = np.linspace(min(strikes_K), max(strikes_K), 100)
plt.plot(strikes_K, vols_K, 'o', label='Vols implicites')
plt.plot(x_K, np.polyval(coef_K, x_K), 'r-', label='Courbe ajustée')
plt.title('Smile COK21 (Mai)')
plt.legend()
plt.show()